In [1]:
import math

# Funcao de input segura
def in_f(msg, unid=""):
    while True:
        try:
            # .format em vez de f-string
            txt = "{} [{}]: ".format(msg, unid)
            val_str = input(txt)
            # Remove virgulas e espacos
            val_str = val_str.replace(',', '.').strip()
            if val_str == "": return None
            return float(val_str)
        except ValueError:
            print("Valor invalido.")

print("--- CALC TRIBOLOGIA 2025 ---")

# --- 0. TIPO ---
while True:
    tipo = input("Tipo? (L=Linear / E=Eliptico): ").upper()
    if tipo in ['L', 'E']: break

# --- 1. GEOMETRIA ---
print("\n[1] GEOMETRIA (Concavo = Negativo)")

Rx1 = in_f("Rx1 (Rolam)", "mm")
Ry1 = 0
if tipo == 'E': Ry1 = in_f("Ry1 (Transv)", "mm")

Rx2 = in_f("Rx2 (Rolam)", "mm")
Ry2 = 0
if tipo == 'E': Ry2 = in_f("Ry2 (Transv)", "mm")

l = 0
if tipo == 'L': l = in_f("Comprim. l", "mm") * 1e-3

# Materiais
E1 = in_f("E1", "GPa")
v1 = in_f("v1", "-")
E2 = in_f("E2", "GPa")
v2 = in_f("v2", "-")

# Calculos Internos SI
Rx1, Rx2 = Rx1*1e-3, Rx2*1e-3
if tipo == 'E': 
    Ry1, Ry2 = Ry1*1e-3, Ry2*1e-3

# E equivalente
E_star = 1 / ((1-v1**2)/(E1*1e9) + (1-v2**2)/(E2*1e9))

# Rx Eq
# Tratamento para zero para evitar erro
irx1 = 1/Rx1 if Rx1!=0 else 0
irx2 = 1/Rx2 if Rx2!=0 else 0
sum_rx = irx1 + irx2
if sum_rx != 0:
    Rx_eq = 1 / (0.5 * sum_rx)
else:
    Rx_eq = 1e9 # Infinito simulado

if tipo == 'E':
    iry1 = 1/Ry1 if Ry1!=0 else 0
    iry2 = 1/Ry2 if Ry2!=0 else 0
    sum_ry = iry1 + iry2
    if sum_ry != 0:
        Ry_eq = 1 / (0.5 * sum_ry)
    else:
        Ry_eq = 1e9
    
    arx, ary = abs(Rx_eq), abs(Ry_eq)
    if arx <= ary: 
        A, B = 1/arx, 1/ary
    else: 
        A, B = 1/ary, 1/arx
    
    ratio = A/B
    sum_AB = A+B
    
    print("Rx_eq: {:.4f} mm".format(Rx_eq*1000))
    print("Ry_eq: {:.4f} mm".format(Ry_eq*1000))
    print("A: {:.4f}".format(A))
    print("B: {:.4f}".format(B))
    print("A/B: {:.4f}".format(ratio))
else:
    # Linear
    A = 1/abs(Rx_eq)
    B = 0
    print("Rx_eq: {:.4f} mm".format(Rx_eq*1000))
    print("A: {:.4f}".format(A))

# --- A. HERTZ ---
print("\n[A] HERTZ")
if tipo == 'E':
    delta = in_f("Penetracao", "um") * 1e-6
    Ca = in_f("Ca", "-")
    Cd = in_f("Cd", "-")
    k = in_f("k (a/b)", "-")
    
    num = delta * math.pi * Ca * E_star
    den = Cd * (sum_AB * E_star)**(1/3)
    Fn = (num/den)**1.5
    
    a = Ca * (Fn/(sum_AB*E_star))**(1/3)
    b = a/k
    Pmax = (3*k*Fn)/(2*math.pi*a**2)
    Ac = math.pi*a*b
else:
    mode = input("Dado? (F=Fn, P=Pmax): ").upper()
    if mode == 'P':
        Pm_GPa = in_f("Pmax", "GPa")
        Pmax = Pm_GPa * 1e9
        Fn = (Pmax**2 * math.pi * l * abs(Rx_eq)) / (2*E_star)
    else:
        Fn = in_f("Fn", "N")
        term_p = (2 * Fn * E_star) / (math.pi * l * abs(Rx_eq))
        Pmax = math.sqrt(term_p)
    
    a = (abs(Rx_eq)*Pmax)/E_star
    b = 0
    Ac = 2*a*l

print("Fn: {:.2f} N".format(Fn))
print("Pmax: {:.3f} GPa".format(Pmax/1e9))
print("a: {:.4f} mm".format(a*1000))
print("Area: {:.4f} mm2".format(Ac*1e6))

# --- B. LUBRIFICACAO ---
print("\n[B] LUBRIFICACAO")
U1 = in_f("U1", "m/s")
U2 = in_f("U2", "m/s")
Usum = U1+U2
DU = abs(U1-U2)

eta0 = in_f("eta0", "Pa.s")
alpha = in_f("alpha", "Pa-1")
beta = in_f("beta", "C-1")
koil = in_f("K_oleo", "W/mK")

Rq1 = in_f("Rq1", "um")
Rq2 = in_f("Rq2", "um")
sigma = math.sqrt(Rq1**2 + Rq2**2)*1e-6
Ra_m = sigma / math.sqrt(2)

if tipo == 'E':
    C0_in = in_f("C0 (Enter p/ calc)", "-")
    if C0_in: 
        C0 = C0_in
    else:
        rr = abs(Ry_eq)/abs(Rx_eq)
        if abs(Rx_eq) > abs(Ry_eq): rr = abs(Rx_eq)/abs(Ry_eq)
        C0 = 1 - 0.61*math.exp(-0.752*rr**0.64)
        print("C0 auto: {:.4f}".format(C0))
        
    num_h = (eta0*Usum)**0.67 * alpha**0.53 * abs(Rx_eq)**0.464
    den_h = Fn**0.067 * E_star**0.073
    hiso = 1.165 * C0 * num_h / den_h
else:
    t_visc = (alpha*eta0*Usum)**0.727
    t_geom = abs(Rx_eq)**0.364 * (l*E_star)**0.091
    hiso = 0.975 * t_visc * t_geom / (Fn**0.091)

# Correcao Termica
L = (beta*eta0*Usum**2)/koil
if Usum > 0:
    Ve = DU/Usum
else:
    Ve = 0
phi_t = 1.0 / (1 + 0.1*(1+14.8*Ve**0.83)*L**0.64)

hcorr = hiso * phi_t
Lambda = hcorr/sigma

print("L param: {:.4f}".format(L))
print("Phi_T: {:.4f}".format(phi_t))
print("h_iso: {:.3f} um".format(hiso*1e6))
print("h_corr: {:.3f} um".format(hcorr*1e6))
print("Lambda: {:.3f}".format(Lambda))

# --- C. TEMPERATURA ---
print("\n[C] TEMPERATURAS")
T0 = in_f("T0", "C")
PT1 = in_f("PT1", "SI")
PT2 = in_f("PT2", "SI")

mu = 0
if DU > 1e-9:
    if tipo == 'E':
        tk = (29.66e-3*Fn)/(eta0*2*b*DU*Usum**2)
        if tk>0: 
            mu = 0.0127*math.log10(tk)
        else: 
            mu = 0.05
        print("Mu (Kelley): {:.4f}".format(mu))
    else:
        # Linear
        den_iso = eta0*l*abs(Rx_eq)*Usum
        term_iso = (Fn*Ra_m)/den_iso
        mu_iso = 0.0254 * (term_iso**0.25)
        
        tm1 = (Fn/l) * (1/(abs(Rx_eq)*Usum))
        mu_mich = 0.0778 * tm1**0.2 * (1/eta0)**0.05 * Ra_m**0.25
        
        print("1. ISO: {:.4f}".format(mu_iso))
        print("2. Mich: {:.4f}".format(mu_mich))
        print("3. Manual")
        
        while True:
            op = input("Opcao (1/2/3): ")
            if op == '1': 
                mu = mu_iso
                break
            elif op == '2': 
                mu = mu_mich
                break
            elif op == '3':
                mu = in_f("Valor Mu", "-")
                break
        print("Mu usado: {:.4f}".format(mu))

# T_Flash
denf = PT1*math.sqrt(abs(U1)) + PT2*math.sqrt(abs(U2))
if denf==0: denf=1

if tipo == 'E':
    wid = 2*b
    # Formula Eliptica
    Tf = 0.893 * (mu*DU/denf) * (Fn/wid)**0.75 * (E_star/abs(Rx_eq))**0.25
else:
    wid = l
    # Formula Linear
    Tf = 0.893 * (mu*DU/denf) * (Fn/wid)**0.75 * (E_star/abs(Rx_eq))**0.25

print("T_Flash: {:.2f} C".format(Tf))

# T_Centro
print("Calc. TC (E=Engr, R=Rolam)")
st = input("Metodo? ").upper()

if st == 'R':
    if tipo == 'E':
        leq = 2*b 
    else:
        leq = l
    tr = ((Fn/leq)*(E_star/abs(Rx_eq)))**0.5
    Tc = 0.157 * tr * (mu*DU*hcorr)/koil
else:
    Z0 = in_f("Z0 (Roelands)", "-")
    if tipo == 'E':
        pm = Fn/(math.pi*a*b)
    else:
        pm = Fn/(2*a*l)
    print("pm: {:.3f} GPa".format(pm/1e9))
    
    fp = (1 + pm/0.196e9)**Z0 - 1
    ep = eta0 * math.exp((math.log(eta0)+9.67)*fp)
    aln = (beta * DU**2 * ep)/(8*koil) + 1
    Tc = (1/beta)*math.log(aln)

print("T_Centro: {:.2f} C".format(Tc))
print("T_MAX: {:.2f} C".format(T0+Tf+Tc))

# --- D. PREVISAO ---
print("\n[D] PREVISAO T")
Lt = in_f("Lambda Alvo", "-")
if Lt:
    v0 = in_f("Visc. Cin. T0", "cSt")
    t_par = in_f("Param 't'", "-")
    
    if tipo == 'L':
        ph = 0.727 
        nexp = 2.0 
    else:
        ph = 0.67
        nexp = 1.5
    
    nt = v0 * ((Lt/Lambda)**(nexp/(ph*(t_par+1))))
    print("Visc Alvo: {:.2f} cSt".format(nt))

--- CALC TRIBOLOGIA 2025 ---


Tipo? (L=Linear / E=Eliptico):  L



[1] GEOMETRIA (Concavo = Negativo)


Rx1 (Rolam) [mm]:  2.5
Rx2 (Rolam) [mm]:  2.5
Comprim. l [mm]:  8
E1 [GPa]:  210
v1 [-]:  0.3
E2 [GPa]:  210
v2 [-]:  0.3


Rx_eq: 2.5000 mm
A: 400.0000

[A] HERTZ


KeyboardInterrupt: Interrupted by user